## Importing Necessary Libraries

In [95]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
import torch

In [96]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Loading Data and Investing

In [97]:
posts_path =r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv"
df_posts = pd.read_csv(r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv")
# Drop column type and image
df_posts = df_posts.drop(columns=['type', 'image'])

In [98]:
comment_path = r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts_comments.csv"
df_comments = pd.read_csv(comment_path, delimiter=";") 
# Drop first column
df_comments = df_comments.iloc[:, 1:]
df_comments.head()

,post_id,comments,category
0,1,"I have a proposal for u, send me a DM please!",0.0
1,1,Woaaaah,0.0
2,1,:smiling_face_with_heart-eyes::smiling_face_wi...,1.0
3,1,:red_heart:️:red_heart:️:red_heart:️,1.0
4,1,:smiling_face_with_heart-eyes::smiling_face_wi...,0.0


## Data Pre-Processing

In [99]:
def overall(df):
    print ("Rows : " ,df.shape[0])
    print ("Columns : " ,df.shape[1])
    print ("\nFeatures : \n" ,df.columns.tolist())
    print ("\nMissing values : ", df.isnull().sum().values.sum())
    print ("\nUnique values : \n", df.nunique())
    
overall(df_posts)

Rows :  827
Columns :  7

Features : 
 ['post_id', 'timestamp', 'ownerUsername', 'caption', 'hashtags', 'likesCount', 'commentsCount']

Missing values :  360

Unique values : 
 post_id          827
timestamp        827
ownerUsername     48
caption          777
hashtags         359
likesCount       787
commentsCount    358
dtype: int64


In [100]:
overall(df_comments)

Rows :  5432
Columns :  3

Features : 
 ['post_id', 'comments', 'category']

Missing values :  0

Unique values : 
 post_id      827
comments    3915
category       2
dtype: int64


#### Check missing value (Post Dataframe)

In [101]:
df_posts.isnull().sum()

post_id            0
timestamp          0
ownerUsername      0
caption            7
hashtags         353
likesCount         0
commentsCount      0
dtype: int64

In [107]:
def load_data(file_path):
    df = pd.read_csv(file_path)
    
    required_columns = ['post_id', 'caption', 'hashtags']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Column {col} not found in the CSV file")
    
    df['post_id'] = df['post_id'].astype(str)
    df['caption'] = df['caption'].astype(str)
    df['hashtags'] = df['hashtags'].astype(str)
    
    df['caption'] = df['caption'].fillna('no_caption')
    df['hashtags'] = df['hashtags'].fillna('no_hashtags')
    
    return df

In [ ]:
def clean_caption(text):
    if pd.isna(text) or text == 'nan':
        return ""
    
    # Chuyển về chuỗi
    text = str(text)
    
    # Loại bỏ URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Loại bỏ ký tự đặc biệt và số, giữ lại emoji
    text = re.sub(r'[^\w\s\U0001F000-\U0001F9FF]', ' ', text)
    
    # Loại bỏ ký tự HTML nếu có
    text = re.sub(r'<.*?>', '', text)
    
    # Chuyển về chữ thường
    text = text.lower()
    
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
def extract_and_replace_emoji(text):
    emojis_list = [c for c in text if c in emoji.EMOJI_DATA]
    
    emoji_descriptions = []
    for e in emojis_list:
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', ' ')
        emoji_descriptions.append(emoji_name)
    

    text_without_emoji = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    return text_without_emoji, emoji_descriptions

In [ ]:
def process_hashtags(text):
    hashtags = re.findall(r'#(\w+)', text)
    
    processed_hashtags = []
    for tag in hashtags:
        words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', tag)
        if not words:  
            words = [tag]
        processed_hashtags.extend([word.lower() for word in words])
    
    clean_text_no_hashtags = re.sub(r'#\w+', '', text)
    
    return clean_text_no_hashtags, processed_hashtags

In [ ]:
def tokenize_and_remove_stopwords(text):
    # Tokenize
    tokens = word_tokenize(text)
    
    # Loại bỏ stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    
    return filtered_tokens

In [ ]:
def train_word2vec(tokenized_texts, vector_size=100, window=5, min_count=1):
    model = Word2Vec(sentences=tokenized_texts, vector_size=vector_size, window=window, min_count=min_count, workers=4)
    model.train(tokenized_texts, total_examples=len(tokenized_texts), epochs=10)
    return model

In [ ]:
def get_bert_embeddings(texts, max_length=128):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')
    
    all_embeddings = []
    
    for text in texts:
        # Tokenize và convert thành input ids
        inputs = tokenizer(text, return_tensors="pt", max_length=max_length, padding="max_length", truncation=True)
        
        # Chạy qua BERT model để lấy embeddings
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Lấy embedding của [CLS] token (đại diện cho toàn bộ câu)
        sentence_embedding = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(sentence_embedding[0])
    
    return np.array(all_embeddings)

In [ ]:
def process_dataframe(df, text_column='caption'):
    if text_column not in df.columns:
        raise ValueError(f"Column {text_column} not found in DataFrame")
    
    df_processed = df.copy()
    
    print("Bước 1: Làm sạch dữ liệu...")
    df_processed['clean_text'] = df_processed[text_column].apply(clean_text)
    
    print("Bước 2: Xử lý emoji...")
    processed_emoji_data = df_processed['clean_text'].apply(extract_and_replace_emoji)
    df_processed['text_no_emoji'] = [item[0] for item in processed_emoji_data]
    df_processed['extracted_emoji'] = [item[1] for item in processed_emoji_data]
    
    print("Bước 3: Xử lý hashtags...")
    processed_hashtag_data = df_processed['text_no_emoji'].apply(process_hashtags)
    df_processed['text_clean'] = [item[0] for item in processed_hashtag_data]
    df_processed['extracted_hashtags'] = [item[1] for item in processed_hashtag_data]
    
    print("Bước 4: Tokenize và loại bỏ stopwords...")
    df_processed['tokens'] = df_processed['text_clean'].apply(tokenize_and_remove_stopwords)
    
    print("Bước 5: Kết hợp tokens...")
    df_processed['combined_tokens'] = df_processed.apply(
        lambda row: row['tokens'] + row['extracted_emoji'] + row['extracted_hashtags'], axis=1
    )
    
    return df_processed

In [ ]:
def create_embeddings(df_processed):
    # Bước 6: Training Word2Vec model
    print("Bước 6: Training Word2Vec model...")
    all_tokenized_texts = df_processed['combined_tokens'].tolist()
    word2vec_model = train_word2vec(all_tokenized_texts)
    
    # Bước 7: Tạo embedding vector cho mỗi post
    print("Bước 7: Tạo Word2Vec embeddings...")
    df_processed['word2vec_embedding'] = df_processed['combined_tokens'].apply(
        lambda tokens: np.mean([word2vec_model.wv[word] for word in tokens if word in word2vec_model.wv], axis=0)
        if tokens and any(word in word2vec_model.wv for word in tokens) else np.zeros(word2vec_model.vector_size)
    )
    
    # Bước 8: BERT embedding
    print("Bước 8: Tạo BERT embeddings...")
    # Chuẩn bị văn bản cho BERT
    df_processed['text_for_bert'] = df_processed.apply(
        lambda row: row['text_clean'] + ' ' + ' '.join(row['extracted_emoji']), axis=1
    )
    
    # Có thể comment phần này nếu không cần BERT hoặc muốn tiết kiệm thời gian
    texts_for_bert = df_processed['text_for_bert'].tolist()
    bert_embeddings = get_bert_embeddings(texts_for_bert)
    
    # Thêm BERT embeddings vào DataFrame
    bert_dim = min(10, bert_embeddings.shape[1])
    for i in range(bert_dim):
        df_processed[f'bert_emb_{i}'] = bert_embeddings[:, i]
    
    return df_processed, word2vec_model